# Coarse-to-fine generation from scratch: codes, levels and parallel passes
**LLMs From Basics · AR.7 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/ar-7-coarse-to-fine.ipynb)

In the lesson, next-concept modeling (NCM) writes text level by level: a few coarse codes first, then finer ones, then tokens, with every position of a level generated in parallel. This notebook is a **toy** of that idea in NumPy (it is not the paper's model). The toy language has a hidden hierarchy, and you will:

1. Learn a 2-level hierarchical code with k-means (level 1: one code per sequence, level 2: one per half-sequence)
2. Fit a model for each level given the coarser ones, and sample all positions of a level in parallel: **3 forward passes** per sequence
3. Build an autoregressive baseline (8 passes) and a simple **coherence** metric
4. Commit $k$ tokens per pass with no hierarchy (the diffusion-style trade-off from AR.6) and compare quality against passes
5. Experiment: delete the middle level and watch quality collapse

Each exercise has a `# TODO` stub followed by a check cell; solutions are at the end. CPU only, a few seconds.

## The toy language

A sequence has 8 tokens, in two halves of 4. Hidden structure: a **theme** $a \in \{0,1,2,3\}$ shared by the whole sequence, and a **variant** $v_h \in \{0,1\}$ for each half $h$. Token $s$ is one of 3 words chosen uniformly from a list that depends on (theme, variant of its half, position in the half). There are $4 \times 2 \times 4 \times 3 = 96$ words, so every word tells you its theme, variant and slot.

A sequence is **coherent** when all 8 tokens share a theme, the 4 tokens of each half share a variant, and each token sits in its own slot. Given the codes (theme, variants), the 8 tokens are independent. That is exactly the situation where generating them in parallel is safe.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

N_THEME, N_VAR, HALF, N_CHOICE, L = 4, 2, 4, 3, 8
V = N_THEME * N_VAR * HALF * N_CHOICE            # 96 words

def word_id(theme, var, slot, choice):
    return ((theme * N_VAR + var) * HALF + slot) * N_CHOICE + choice

def decode(t):
    # token id -> (theme, variant, slot in half)
    return t // 24, (t // 12) % 2, (t // 3) % 4

def sample_data(n, rng):
    theme = rng.integers(N_THEME, size=n)
    var = rng.integers(N_VAR, size=(n, 2))
    choice = rng.integers(N_CHOICE, size=(n, L))
    X = np.zeros((n, L), dtype=int)
    for s in range(L):
        X[:, s] = word_id(theme, var[:, s // HALF], s % HALF, choice[:, s])
    return X, theme, var

X, theme_true, var_true = sample_data(4000, rng)
print(X[:3])
print("theme, variants of the first sequence:", theme_true[0], var_true[0])

## 1. Learn a hierarchical code with k-means

We "tokenize" each sequence into a hierarchy: **level 1** is a single code for the whole sequence (4 clusters), **level 2** is one code per half (8 clusters: a theme plus a variant), and **level 3** is the tokens themselves. Each word has a fixed 10-number embedding `E[w]` (a strong theme direction, a medium variant direction, a weak slot direction and a little per-word noise, standing in for a pretrained embedding table). A sequence or half is represented by its **chunk embedding**: the mean of its tokens' embeddings.

Write `kmeans` (the initial centres come from `kmeanspp_init`) and `learn_codes`. For level 2, pool both halves into one dataset so both halves share the same code labels.

In [ ]:
def make_embeddings(rng, noise=0.3):
    E = np.zeros((V, 10))
    for w in range(V):
        th, va, sl = decode(w)
        E[w, th] = 2.0
        E[w, 4 + va] = 1.0
        E[w, 6 + sl] = 0.5
    return E + noise * rng.normal(size=E.shape)

E = make_embeddings(np.random.default_rng(123))

def chunk_embed(tokens):
    # tokens: (n, m) ints -> (n, 10): mean of the word embeddings of each row
    return E[tokens].mean(axis=1)

def kmeanspp_init(Z, k, rng):
    centers = [Z[rng.integers(len(Z))]]
    for _ in range(k - 1):
        d2 = np.min(((Z[:, None, :] - np.array(centers)[None]) ** 2).sum(-1), axis=1)
        centers.append(Z[rng.choice(len(Z), p=d2 / d2.sum())])
    return np.array(centers)

def inertia(Z, labels, centers):
    return float(((Z - centers[labels]) ** 2).sum())

In [ ]:
def kmeans(Z, k, rng, iters=25):
    centers = kmeanspp_init(Z, k, rng)
    for _ in range(iters):
        # TODO: assign every row of Z to its nearest centre (squared distance),
        #       then move each centre to the mean of its rows (keep the old centre if a cluster is empty)
        pass
    labels = None  # TODO: final assignment
    return labels, centers

def learn_codes(X, rng, restarts=4):
    # TODO: level 1: k-means with k=4 on chunk_embed(X) -> c1, shape (n,)
    #       level 2: pool both halves into one (2n, 10) array of chunk embeddings, k-means with k=8 -> c2, shape (n, 2)
    #       run `restarts` times per level and keep the run with the lowest inertia
    raise NotImplementedError

In [ ]:
def purity(labels, truth):
    # fraction of points that sit in the majority true class of their cluster (1.0 = perfect recovery)
    labels, truth = np.asarray(labels), np.asarray(truth)
    return sum(np.bincount(truth[labels == c]).max() for c in np.unique(labels)) / len(labels)

sub = np.arange(3000)
c1, c2 = learn_codes(X[sub], np.random.default_rng(1))
assert c1.shape == (3000,) and c2.shape == (3000, 2)
assert len(np.unique(c1)) == 4 and len(np.unique(c2)) == 8
assert purity(c1, theme_true[sub]) == 1.0, "level 1 should recover the theme"
true_c2 = theme_true[sub, None] * N_VAR + var_true[sub]               # (theme, variant) for each half
assert purity(c2.ravel(), true_c2.ravel()) == 1.0, "level 2 should recover (theme, variant) per half"
print("Exercise 1 passed: the learned codes match the hidden structure exactly.")

## 2. Fit a model for each level and sample a level in parallel

With the codes in hand, estimate three tables by counting:

- `T1`: the prior over level-1 codes, shape `(4,)`
- `T2[i, j] = P(level-2 code = j | level-1 code = i)`, pooled over both halves, shape `(4, 8)`
- `T3[j, s, w] = P(token = w | level-2 code = j, slot s within the half)`, shape `(8, 4, 96)`, with a tiny smoothing constant

Then `sample_ncm` generates $n$ sequences in **3 passes**: sample level 1 for everyone, then sample **both** level-2 codes of every sequence at once, then sample **all 8 tokens** of every sequence at once. Within a pass the positions are drawn independently, given everything coarser.

In [ ]:
def sample_categorical(P, rng):
    # P: (n, K) rows of probabilities -> (n,) sampled indices
    u = rng.random((len(P), 1))
    return np.minimum((P.cumsum(axis=1) < u).sum(axis=1), P.shape[1] - 1)

In [ ]:
def fit_tables(c1, c2, X, alpha=1e-3):
    # TODO: count and normalise
    #   T1 (4,): frequency of each level-1 code
    #   T2 (4, 8): rows normalised counts of (c1, c2[:, h]) for both halves h
    #   T3 (8, 4, 96): counts of (c2[:, h], slot, token) for both halves, plus alpha, rows normalised over tokens
    raise NotImplementedError

def sample_ncm(tables, n, rng):
    T1, T2, T3 = tables
    # TODO: pass 1: c1 ~ T1 (shape (n,))
    #       pass 2: both level-2 codes ~ T2[c1], drawn independently (shape (n, 2))
    #       pass 3: all 8 tokens, token s ~ T3[c2[:, s // 4], s % 4], drawn independently
    passes = 3
    raise NotImplementedError

In [ ]:
c1_all, c2_all = learn_codes(X, np.random.default_rng(2))
tables = fit_tables(c1_all, c2_all, X)
T1, T2, T3 = tables
assert T1.shape == (4,) and T2.shape == (4, 8) and T3.shape == (8, 4, 96)
assert np.allclose(T1.sum(), 1) and np.allclose(T2.sum(1), 1) and np.allclose(T3.sum(2), 1)
assert (T2 > 0).sum(axis=1).tolist() == [2, 2, 2, 2], "each theme should use exactly 2 level-2 codes"
Xs, passes = sample_ncm(tables, 2000, np.random.default_rng(3))
assert Xs.shape == (2000, 8) and passes == 3
assert Xs.min() >= 0 and Xs.max() < V
print("Exercise 2 passed: 3 passes, sampled", Xs.shape[0], "sequences. First:", Xs[0])

## 3. A coherence metric and an autoregressive baseline

Write `coherent(X)`: for each sequence, `True` if all tokens share one theme, the tokens of each half share one variant, and every token's slot equals its position in its half (`s % 4`). Then fit an **order-2 autoregressive** model: $P(x_s \mid x_{s-2}, x_{s-1})$ by counting, with two start-of-sequence symbols (id 96) so the first tokens have a context. Sampling it takes **8 passes**, one per token.

In [ ]:
def coherent(X):
    # TODO: return a boolean array (n,). Use decode(X) -> theme, variant, slot arrays of shape (n, 8).
    raise NotImplementedError

BOS = V   # start-of-sequence symbol, id 96

def fit_ar(X, alpha=1e-3):
    # TODO: counts[p2, p1, w] for every position s, with p2, p1 the two previous tokens (BOS before the start).
    #       Return probabilities of shape (V + 1, V + 1, V): add alpha, then normalise over w.
    raise NotImplementedError

def sample_ar(P, n, rng):
    # TODO: 8 sequential passes; at pass s sample x_s from P[x_{s-2}, x_{s-1}]
    raise NotImplementedError

In [ ]:
hand = np.array([[word_id(1, 0, s % 4, 0) if s < 4 else word_id(1, 1, s % 4, 2) for s in range(8)],   # coherent
                 [word_id(1, 0, s % 4, 0) if s < 4 else word_id(2, 1, s % 4, 2) for s in range(8)],   # theme changes
                 [word_id(1, 0, s % 4, 0) if s != 2 else word_id(1, 1, 2, 0) for s in range(8)]])     # variant breaks inside a half
assert coherent(hand).tolist() == [True, False, False], coherent(hand)
assert coherent(X).all(), "training data is coherent by construction"

P_ar = fit_ar(X)
assert P_ar.shape == (V + 1, V + 1, V) and np.allclose(P_ar.sum(2), 1)
Xa, passes_ar = sample_ar(P_ar, 2000, np.random.default_rng(4))
assert Xa.shape == (2000, 8) and passes_ar == 8

coh_ncm, coh_ar = coherent(Xs).mean(), coherent(Xa).mean()
themes_ncm = np.bincount(decode(Xs)[0][:, 0], minlength=4) / len(Xs)
print(f"NCM-style: {coh_ncm:.3f} coherent in 3 passes   AR order-2: {coh_ar:.3f} coherent in {passes_ar} passes")
print("theme frequencies of NCM samples:", themes_ncm)
assert coh_ncm > 0.99 and coh_ar > 0.95
assert np.abs(themes_ncm - 0.25).max() < 0.05, "samples should cover all themes (no mode collapse)"
print("Exercise 3 passed.")

## 4. No hierarchy: commit $k$ tokens per pass

Now remove the codes and try the diffusion-style shortcut from AR.6: in each pass, pick $k$ still-unknown positions at random and commit them **independently**, each from its own marginal given what is already known. To keep it about the *method* and not about training, we give you a perfect model, `oracle_marginals(tokens, known)`, which returns each position's exact marginal distribution given the committed tokens (shape `(8, 96)`).

Write `parallel_sample(k, n, rng)`. It needs $\lceil 8/k \rceil$ passes. Compare coherence against passes with the hierarchy's single point.

In [ ]:
COMBOS = [(a, v0, v1) for a in range(N_THEME) for v0 in range(N_VAR) for v1 in range(N_VAR)]   # 16 hidden states

def oracle_marginals(tokens, known):
    # tokens: (8,) ints (entries where known is False are ignored); known: (8,) bool
    post = np.zeros(len(COMBOS))
    for ci, (a, v0, v1) in enumerate(COMBOS):
        bad = 0
        for s in np.flatnonzero(known):
            th, va, sl = decode(tokens[s])
            bad += (th != a) + (va != (v0 if s < HALF else v1)) + (sl != s % HALF)
        post[ci] = 1e-9 ** bad                # soft evidence: a confused model (conflicting tokens) still has an opinion
    post /= post.sum()
    marg = np.zeros((L, V))
    for ci, (a, v0, v1) in enumerate(COMBOS):
        for s in range(L):
            for c in range(N_CHOICE):
                marg[s, word_id(a, v0 if s < HALF else v1, s % HALF, c)] += post[ci] / N_CHOICE
    return marg

m0 = oracle_marginals(np.zeros(8, dtype=int), np.zeros(8, dtype=bool))
assert np.allclose(m0.sum(1), 1) and np.isclose(m0[0].max(), 1 / 24)

In [ ]:
def parallel_sample(k, n, rng):
    out = np.zeros((n, L), dtype=int)
    passes = int(np.ceil(L / k))
    for i in range(n):
        known = np.zeros(L, dtype=bool)
        tokens = np.zeros(L, dtype=int)
        for _ in range(passes):
            # TODO: choose up to k unknown positions at random, compute oracle_marginals(tokens, known) once,
            #       and sample each chosen position independently from its own marginal; mark them known
            pass
        out[i] = tokens
    return out, passes

In [ ]:
curve = []
for k in [8, 4, 2, 1]:
    Xp, p = parallel_sample(k, 300, np.random.default_rng(10 + k))
    curve.append((p, coherent(Xp).mean()))
    print(f"{k} tokens per pass: {p} passes, {curve[-1][1]:.3f} coherent")
assert curve[0][0] == 1 and curve[-1][0] == 8
assert curve[-1][1] == 1.0, "one token per pass with a perfect model is always coherent"
assert curve[0][1] < 0.02, "committing all 8 tokens at once from their marginals is almost never coherent"
assert curve[0][1] <= curve[1][1] + 0.05 and curve[1][1] < curve[3][1]

plt.figure(figsize=(5.5, 3.4))
plt.plot(*zip(*curve), "o-", color="tab:orange", label="no hierarchy, k tokens per pass")
plt.plot([3], [coherent(Xs).mean()], "*", ms=14, color="tab:blue", label="3-level hierarchy")
plt.plot([8], [coherent(Xa).mean()], "s", color="tab:green", label="autoregressive (order 2)")
plt.xlabel("forward passes per sequence"); plt.ylabel("coherent fraction"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Exercise 4 passed.")

## Experiment: what if a level is missing?

Delete the middle level: sample the theme (level 1), then draw all 8 tokens in one parallel pass given only the theme. That is 2 passes. Fit `P(token | theme, slot)` by counting and measure coherence. Then count positions processed per sample, as in the lesson: an autoregressive model with a cache processes 8 positions, the 3-level toy processes $1 + 2 + 8 = 11$, and the 2-level version $1 + 8 = 9$.

In [ ]:
# level-1-only model: P(token | theme code, slot s in 0..7), fit by counting
T_theme = np.full((4, L, V), 1e-3)
for s in range(L):
    np.add.at(T_theme, (c1_all, s, X[:, s]), 1.0)
T_theme /= T_theme.sum(axis=2, keepdims=True)

r = np.random.default_rng(5)
c1_s = sample_categorical(np.tile(T1, (2000, 1)), r)
X2 = np.stack([sample_categorical(T_theme[c1_s, s], r) for s in range(L)], axis=1)
print(f"2 passes (theme, then tokens): {coherent(X2).mean():.3f} coherent   3 passes (theme, variants, tokens): {coherent(Xs).mean():.3f}")
print("positions processed per sample: AR 8, 3-level 1+2+8 = 11, 2-level 1+8 = 9")
assert coherent(X2).mean() < 0.1 < 0.9 < coherent(Xs).mean()
print("Experiment done. The dependencies between tokens (their shared variant) live in level 2; without that level the tokens of a half disagree.")

## Solutions
Run these only after trying the stubs: they redefine the functions above, so re-run the check cells afterwards.

### Solution 1

In [ ]:
def kmeans(Z, k, rng, iters=25):
    centers = kmeanspp_init(Z, k, rng)
    for _ in range(iters):
        d2 = ((Z[:, None, :] - centers[None]) ** 2).sum(-1)
        labels = d2.argmin(1)
        for j in range(k):
            if np.any(labels == j):
                centers[j] = Z[labels == j].mean(0)
    d2 = ((Z[:, None, :] - centers[None]) ** 2).sum(-1)
    return d2.argmin(1), centers

def best_kmeans(Z, k, rng, restarts):
    best = None
    for _ in range(restarts):
        labels, centers = kmeans(Z, k, rng)
        score = inertia(Z, labels, centers)
        if best is None or score < best[0]:
            best = (score, labels)
    return best[1]

def learn_codes(X, rng, restarts=4):
    n = len(X)
    c1 = best_kmeans(chunk_embed(X), 4, rng, restarts)
    halves = np.concatenate([X[:, :HALF], X[:, HALF:]], axis=0)       # (2n, 4)
    c2_pooled = best_kmeans(chunk_embed(halves), 8, rng, restarts)
    c2 = np.stack([c2_pooled[:n], c2_pooled[n:]], axis=1)             # (n, 2)
    return c1, c2

### Solution 2

In [ ]:
def fit_tables(c1, c2, X, alpha=1e-3):
    n = len(X)
    T1 = np.bincount(c1, minlength=4).astype(float)
    T1 /= T1.sum()
    T2 = np.zeros((4, 8))
    T3 = np.zeros((8, HALF, V))
    for h in range(2):
        np.add.at(T2, (c1, c2[:, h]), 1.0)
        for s in range(HALF):
            np.add.at(T3, (c2[:, h], s, X[:, h * HALF + s]), 1.0)
    T2 /= T2.sum(axis=1, keepdims=True)
    T3 += alpha
    T3 /= T3.sum(axis=2, keepdims=True)
    return T1, T2, T3

def sample_ncm(tables, n, rng):
    T1, T2, T3 = tables
    c1 = sample_categorical(np.tile(T1, (n, 1)), rng)                                   # pass 1
    c2 = np.stack([sample_categorical(T2[c1], rng) for _ in range(2)], axis=1)          # pass 2
    X = np.zeros((n, L), dtype=int)
    for s in range(L):                                                                  # pass 3
        X[:, s] = sample_categorical(T3[c2[:, s // HALF], s % HALF], rng)
    return X, 3

### Solution 3

In [ ]:
def coherent(X):
    theme, var, slot = decode(X)
    ok_theme = (theme == theme[:, :1]).all(axis=1)
    ok_slot = (slot == (np.arange(L) % HALF)[None, :]).all(axis=1)
    ok_var = (var[:, :HALF] == var[:, :1]).all(axis=1) & (var[:, HALF:] == var[:, HALF:HALF + 1]).all(axis=1)
    return ok_theme & ok_slot & ok_var

BOS = V   # start-of-sequence symbol, id 96

def fit_ar(X, alpha=1e-3):
    n = len(X)
    pad = np.concatenate([np.full((n, 2), BOS), X], axis=1)               # (n, 10)
    counts = np.zeros((V + 1, V + 1, V))
    for s in range(L):
        np.add.at(counts, (pad[:, s], pad[:, s + 1], pad[:, s + 2]), 1.0)
    counts += alpha
    return counts / counts.sum(axis=2, keepdims=True)

def sample_ar(P, n, rng):
    pad = np.full((n, L + 2), BOS)
    for s in range(L):
        pad[:, s + 2] = sample_categorical(P[pad[:, s], pad[:, s + 1]], rng)
    return pad[:, 2:], L

### Solution 4

In [ ]:
def parallel_sample(k, n, rng):
    out = np.zeros((n, L), dtype=int)
    passes = int(np.ceil(L / k))
    for i in range(n):
        known = np.zeros(L, dtype=bool)
        tokens = np.zeros(L, dtype=int)
        for _ in range(passes):
            marg = oracle_marginals(tokens, known)
            todo = rng.permutation(np.flatnonzero(~known))[:k]
            for s in todo:
                tokens[s] = rng.choice(V, p=marg[s])
            known[todo] = True
        out[i] = tokens
    return out, passes